# Module 9 — CI/CD (GitHub Actions -> Databricks Jobs)
Exam domain: **Databricks Tooling**

Databricks notebook used as the deployable artifact itself: this is the kind of
notebook the `daily_batch_pipeline` job in `databricks.yml` would point at.
Actual CI/CD orchestration lives in GitHub Actions (see the Colab version for
the workflow YAML) and the Databricks CLI — not inside a notebook cell.

In [ ]:
dbutils.widgets.text("run_date", "")
run_date = dbutils.widgets.get("run_date") or "unknown"
print(f"Job task executing for run_date={run_date}")

## Deploying this notebook via Databricks CLI / Asset Bundles
```bash
# Authenticate (service principal recommended for CI)
databricks auth login --host https://<workspace>.cloud.databricks.com

# Validate the bundle before deploying
databricks bundle validate

# Deploy Jobs/DLT pipelines defined in databricks.yml to a target
databricks bundle deploy --target prod

# Trigger a run manually (CI usually lets the schedule do this instead)
databricks bundle run daily_batch_pipeline --target prod
```

## Inspecting a deployed job's run history

In [ ]:
%sql
-- SELECT job_id, run_id, result_state, period_start_time, period_end_time
-- FROM system.lakeflow.job_run_timeline
-- WHERE job_name = 'daily-batch-pipeline'
-- ORDER BY period_start_time DESC
-- LIMIT 10;

## Environment promotion pattern
- `dev` target: `mode: development` — bundle deploys prefix resources with the
  deploying user's name, jobs are paused by default, safe to iterate on.
  everything can be redeployed freely without touching prod resources.
- `prod` target: `mode: production` — requires a fixed run-as identity (a
  service principal, not a personal user), jobs run on their real schedule.
- GitHub Actions deploys to `dev` on every PR (optional) and to `prod` only
  from `main`, exactly as wired in the Colab version's workflow file.